## Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pyo.lib.server import Server

from pysynth.clock import Clock
from pysynth.tempo import Tempo
from pysynth.patches import PatchRack, atmosphere, bass, clock_tick, drone, hat, low_hat

s = Server().boot()
s.start()

tempo = Tempo(bpm=132)
rack = PatchRack()

# single shared pulse every patch (except clock_tick, which is deliberately
# free-running) divides down from, so they lock to a common downbeat instead
# of just a common tempo - see src/pysynth/clock.py
clock = Clock(tempo)
clock.start()

Pyo warning: Portaudio input device `MacBook Pro Microphone` has fewer channels (1) than requested (2).
Pyo warning: Portmidi warning: no midi device found!
Portmidi closed.


## Bass

Rolling 16-step bassline through a resonant, LFO-swept lowpass filter. See `src/pysynth/patches/bass.py`.

In [ ]:
bass_params = {
    # Fundamental frequency of the bassline's root note.
    "root_freq": bass.ROOT_FREQ,  # default 55, range 30-110
    # Filter resonance - higher gives a more squelchy, whistling character.
    "filter_res": 0.9,  # default 0.75, range 0-1
    # Center cutoff the sweep rides on - higher is brighter.
    "filter_base": 900,  # default 900, range 200-2000
    # How far the cutoff sweeps each bar - larger is a more dramatic sweep.
    "filter_range": 400,  # default 400, range 0-1000
    # Output level for this patch (limited via Compress so it won't clip).
    "volume": 0.7,  # default 1.0, range 0-2
}
bass_patch = rack.toggle("bass", bass.build, tempo, clock, **bass_params)

## Atmosphere (FM pad + arpeggiator)

FM pad voice arpeggiated at 8th notes, with a slow amplitude swell and reverb. See `src/pysynth/patches/atmosphere.py`.

In [ ]:
atmosphere_params = {
    # How often the arpeggio advances, in 16th notes - larger is slower.
    "step_division": clock.FOURTH,  # default 4 (FOURTH), range 1-16 (SIXTEENTH-BAR)
    # Base frequency of the arpeggio's root note.
    "arp_root": atmosphere.ARP_ROOT,  # default 220, range 110-440
    # Modulator/carrier ratio in the FM voice - controls harmonic vs dissonant tone.
    "fm_ratio": 0.5012,  # default 0.5012, range 0.1-4
    # FM modulation depth - higher is brighter and buzzier.
    "fm_index": 4,  # default 4, range 0-10
    # Reverb room size - larger is bigger and more distant.
    "reverb_size": 0.9,  # default 0.85, range 0-1
    # Reverb high-frequency damping - higher is darker.
    "reverb_damp": 0.9,  # default 0.6, range 0-1
    # Reverb dry/wet balance (0 = dry, 1 = wet).
    "reverb_bal": 0.9,  # default 0.5, range 0-1
    # Output level for this patch (limited via Compress so it won't clip).
    "volume": 1.0,  # default 1.0, range 0-2
}
atmosphere_patch = rack.toggle("atmosphere", atmosphere.build, tempo, clock, **atmosphere_params)

## Hi-hat

Subtle high-passed noise tick, once per 8th note, for top-end texture. See `src/pysynth/patches/hat.py`.

In [ ]:
hat_params = {
    # High-pass cutoff - higher is thinner and more distant.
    "cutoff_freq": hat.CUTOFF_FREQ,  # default 8000, range 2000-12000
    # Loudness of the raw noise burst before shaping.
    "level": 0.2,  # default 0.5, range 0-1
    # Envelope length - shorter is a tighter click, longer is more of a hiss.
    "decay": 0.2,  # default 0.2, range 0.05-1
    # Output level for this patch (limited via Compress so it won't clip).
    "volume": 1.0,  # default 1.0, range 0-2
}
hat_patch = rack.toggle("hat", hat.build, tempo, clock, **hat_params)

## Low hat

Darker noise tick, once per quarter note, as a rarer, dubbier accent. See `src/pysynth/patches/low_hat.py`.

In [ ]:
low_hat_params = {
    # High-pass cutoff - lower than the main hat for a darker tone.
    "cutoff_freq": low_hat.CUTOFF_FREQ,  # default 3000, range 1000-6000
    # Loudness of the raw noise burst before shaping.
    "level": 0.3,  # default 0.3, range 0-1
    # Envelope length - longer than the main hat for a dubbier tail.
    "decay": 0.5,  # default 0.5, range 0.05-1
    # Output level for this patch (limited via Compress so it won't clip).
    "volume": 1.0,  # default 1.0, range 0-2
}
low_hat_patch = rack.toggle("low_hat", low_hat.build, tempo, clock, **low_hat_params)

## Clock tick

Pink Floyd "Time"-style clock shop: four noise ticks, each on its own free-running period in real seconds rather than the tempo grid, so they drift past each other and never share a downbeat — a quiet background texture, not a groove element. See `src/pysynth/patches/clock_tick.py`.

In [ ]:
clock_tick_params = {
    # Overall level of the four summed ticks.
    "level": clock_tick.OVERALL_LEVEL,  # default 0.5, range 0-1
    # Resonance of the woodblock tock - higher is more tonal.
    "wood_q": 3,  # default 3, range 1-10
    # Resonance of the glassy tick - higher is more ringing.
    "glass_q": 6,  # default 6, range 1-15
    # Output level for this patch (limited via Compress so it won't clip).
    "volume": 1.0,  # default 1.0, range 0-2
}
clock_tick_patch = rack.toggle("clock_tick", clock_tick.build, tempo, **clock_tick_params)

## Drone

Slow-winding FM drone: note changes once every 8 bars, with a continuously drifting timbre. See `src/pysynth/patches/drone.py`.

In [ ]:
drone_params = {
    # Fundamental frequency the drone glides between.
    "root_freq": drone.DRONE_ROOT,  # default 110, range 55-220
    # Reverb room size - larger is more smeared and cavernous.
    "reverb_size": 0.9,  # default 0.9, range 0-1
    # Reverb high-frequency damping - lower keeps the tail bright and shimmering.
    "reverb_damp": 0.3,  # default 0.3, range 0-1
    # Reverb dry/wet balance - high here so the drone sits mostly in the reverb.
    "reverb_bal": 0.9,  # default 0.9, range 0-1
    # Output level for this patch (limited via Compress so it won't clip).
    "volume": 1.0,  # default 1.0, range 0-2
}
drone_patch = rack.toggle("drone", drone.build, tempo, clock, **drone_params)

## Stop

Each patch can be stopped/restarted on its own by rerunning `rack.stop("<name>")` / re-running its build cell. This cell stops everything at once.

In [ ]:
agr

In [ ]:
rack.stop_all()

<!-- ## Panic reset

If a sound is stuck playing and `rack.stop_all()` doesn't touch it — most likely it was started before this notebook adopted `PatchRack`, so nothing currently holds a reference to it — the fix is to restart the audio server outright. This tears down the entire audio graph regardless of what Python still references, so you'll need to rerun every patch cell afterwards (they'll pick up `rack` again since that variable survives). -->

In [ ]:
# s.stop()
# s.shutdown()
# s.boot()
# s.start()

In [ ]:
# s.stop()